# 11 · Employee-Level Skills Inventory (Synthetic MVP Model)

**Project:** Enterprise HR AI  

> ### ⚠️ PROMINENT DATA INTEGRITY WARNING
> **SYNTHETIC DATA — employee current-skill possession was not present in any source file and has been simulated using a tenure/training-based heuristic for MVP demonstration purposes only. This must NOT be presented to stakeholders as real observed skill data. Real deployment requires an actual skills inventory (HRIS export, LMS completion records, or self-assessment survey).**

---

---
## Step 1 · Honest Source Data Audit: Does Employee-Level Skill Data Exist?

In [1]:
import pandas as pd
import numpy as np
import os

PROC = os.path.join('..', 'data', 'processed')

att_path = os.path.join(PROC, 'employee_attrition_processed.csv')
eng_path = os.path.join(PROC, 'engagement_processed.csv')
rsp_path = os.path.join(PROC, 'role_skill_profiles.csv')

df_att = pd.read_csv(att_path)
df_eng = pd.read_csv(eng_path)
df_rsp = pd.read_csv(rsp_path)

print('=== COLUMN AUDIT FOR REAL EMPLOYEE-LEVEL SKILLS DATA ===\n')

print(f'1. employee_attrition_processed.csv ({df_att.shape[1]} columns):')
print(list(df_att.columns))
att_skill_cols = [c for c in df_att.columns if 'skill' in c.lower() or 'competenc' in c.lower()]
print(f'   Skill-related columns found: {att_skill_cols}\n')

print(f'2. engagement_processed.csv ({df_eng.shape[1]} columns):')
print(list(df_eng.columns))
eng_skill_cols = [c for c in df_eng.columns if 'skill' in c.lower() or 'competenc' in c.lower()]
print(f'   Skill-related columns found: {eng_skill_cols}\n')

print(f'3. role_skill_profiles.csv ({df_rsp.shape[1]} columns):')
print(list(df_rsp.columns))
rsp_skill_cols = [c for c in df_rsp.columns if 'skill' in c.lower()]
print(f'   Skill-related columns found: {rsp_skill_cols}\n')

print('=' * 80)
print('EXPLICIT AUDIT CONCLUSION:')
print('Does a real employee-level current-skills column exist anywhere in the raw or processed data?')
print('ANSWER: NO.')
print('- employee_attrition has career/demographic fields, but zero skill inventory.')
print('- engagement_processed tracks training courses (name, cost, duration), but no skill mastery.')
print('- role_skill_profiles contains ROLE-LEVEL required O*NET skills, but no employee-level possession.')
print('=' * 80)

=== COLUMN AUDIT FOR REAL EMPLOYEE-LEVEL SKILLS DATA ===

1. employee_attrition_processed.csv (35 columns):
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']
   Skill-related columns found: []

2. engagement_processed.csv (28 columns):
['Employee ID', 'StartDate', 'Title', 'BusinessUnit', 'EmployeeStatus', 'EmployeeType', 'PayZone', 'EmployeeClassificationType', 'DepartmentType', 'Division', 'DOB', 'State', 'GenderCode', 'RaceD

---
## Step 2 · Synthetic Skill Generation Heuristic (MVP Model)

To enable downstream capability matching and gap analysis without fabricating ungrounded data:
1. **Role Benchmark Mapping:** Each employee inherits their job role's benchmark skills from `role_skill_profiles.csv` (top 5 essential skills + top 5 software tools).
2. **Empirical Probability Formula:** Skill possession is simulated using an auditable, tenure/training-conditioned heuristic:
   $$\text{possession\_probability} = \min(0.30 + 0.05 \times \text{YearsAtCompany} + 0.05 \times \text{TrainingTimesLastYear}, 0.95)$$
   - Base probability = 30% (new hire baseline)
   - +5% per year at company (tenure/on-the-job mastery)
   - +5% per training session completed in the last year (active upskilling)
   - Capped at 95% (no employee is synthetically assumed to have 100% mastery)
3. **Manager Role Exclusion:** For the 102 employees with `JobRole == 'Manager'`, skill simulation is **skipped entirely** and mapped to `skill_name = 'N/A - Department-level analysis only'`, adhering to the architectural decision from Step 10 & 11.

In [2]:
# Parse required skills per role from role_skill_profiles.csv
role_skills_dict = {}
for _, r in df_rsp.iterrows():
    role = r['ibm_job_role']
    if r['match_confidence'] == 'very_low':
        continue
    ess_items = [s.split('(')[0].strip() for s in r['top5_essential_skills'].split('|')]
    sw_items = [s.strip() for s in r['top5_software_tools'].split('|')]
    role_skills_dict[role] = (
        [(s, 'essential') for s in ess_items] +
        [(s, 'software') for s in sw_items]
    )

print(f'Parsed benchmarks for {len(role_skills_dict)} job roles (Manager excluded from O*NET profiles).')

# Set fixed random seed for 100% reproducibility
np.random.seed(42)

synthetic_rows = []
audit_examples = []

for _, emp in df_att.iterrows():
    emp_id = emp['EmployeeNumber']
    role = emp['JobRole']
    tenure = emp['YearsAtCompany']
    training = emp['TrainingTimesLastYear']
    
    # Manager Role Exclusion
    if role == 'Manager':
        synthetic_rows.append({
            'EmployeeNumber': emp_id,
            'skill_name': 'N/A - Department-level analysis only',
            'skill_type': 'N/A',
            'has_skill': 0
        })
        continue
        
    # Compute formula
    prob = min(0.30 + 0.05 * tenure + 0.05 * training, 0.95)
    req_skills = role_skills_dict[role]
    
    possessed = []
    for skill_name, skill_type in req_skills:
        has = int(np.random.rand() < prob)
        synthetic_rows.append({
            'EmployeeNumber': emp_id,
            'skill_name': skill_name,
            'skill_type': skill_type,
            'has_skill': has
        })
        if has:
            possessed.append(skill_name)
            
    if len(audit_examples) < 5:
        audit_examples.append({
            'EmployeeNumber': emp_id,
            'JobRole': role,
            'YearsAtCompany': tenure,
            'TrainingTimesLastYear': training,
            'possession_probability': round(prob, 4),
            'skills_possessed_count': len(possessed),
            'total_skills': len(req_skills),
            'skills_possessed': possessed
        })

df_synthetic = pd.DataFrame(synthetic_rows)
print(f'Total synthetic records generated: {len(df_synthetic):,}')
print(f'Total distinct employees covered: {df_synthetic["EmployeeNumber"].nunique():,}')

Parsed benchmarks for 8 job roles (Manager excluded from O*NET profiles).
Total synthetic records generated: 13,782
Total distinct employees covered: 1,470


---
## Step 3 · Auditing Heuristic Behavior: 5 Example Employees

In [3]:
print('=== 5 AUDIT EXAMPLES: HEURISTIC PROBABILITY & SKILL LIST ===')
for ex in audit_examples:
    print(f'\nEmployee #{ex["EmployeeNumber"]} ({ex["JobRole"]})')
    print(f'  Tenure: {ex["YearsAtCompany"]} years | Training Sessions Last Year: {ex["TrainingTimesLastYear"]}')
    print(f'  Possession Probability: min(0.30 + 0.05*{ex["YearsAtCompany"]} + 0.05*{ex["TrainingTimesLastYear"]}, 0.95) = {ex["possession_probability"]:.2f}')
    print(f'  Skills Possessed: {ex["skills_possessed_count"]} of {ex["total_skills"]} required')
    print(f'  Possessed Skills: {ex["skills_possessed"]}')

=== 5 AUDIT EXAMPLES: HEURISTIC PROBABILITY & SKILL LIST ===

Employee #1 (Sales Executive)
  Tenure: 6 years | Training Sessions Last Year: 0
  Possession Probability: min(0.30 + 0.05*6 + 0.05*0, 0.95) = 0.60
  Skills Possessed: 5 of 10 required
  Possessed Skills: ['Active Listening', 'Critical Thinking', 'Monitoring', 'Adobe Acrobat', 'Adobe Creative Cloud software']

Employee #2 (Research Scientist)
  Tenure: 10 years | Training Sessions Last Year: 3
  Possession Probability: min(0.30 + 0.05*10 + 0.05*3, 0.95) = 0.95
  Skills Possessed: 9 of 10 required
  Possessed Skills: ['Critical Thinking', 'Active Listening', 'Speaking', 'Active Learning', 'Amazon DynamoDB', 'Amazon Elastic Compute Cloud EC2', 'Amazon Redshift', 'Amazon Web Services AWS CloudFormation', 'Amazon Web Services AWS software']

Employee #4 (Laboratory Technician)
  Tenure: 0 years | Training Sessions Last Year: 3
  Possession Probability: min(0.30 + 0.05*0 + 0.05*3, 0.95) = 0.45
  Skills Possessed: 5 of 10 required

---
## Step 4 · Verify Manager Role Exclusion

In [4]:
mgr_ids = df_att[df_att['JobRole'] == 'Manager']['EmployeeNumber'].tolist()
mgr_records = df_synthetic[df_synthetic['EmployeeNumber'].isin(mgr_ids)]

print('=== MANAGER ROLE EXCLUSION VERIFICATION ===')
print(f'Expected Manager headcount: 102')
print(f'Total Manager records in synthetic table: {len(mgr_records)}')
print(f'Distinct skill names for Managers: {mgr_records["skill_name"].unique().tolist()}')
print(f'Distinct skill types for Managers: {mgr_records["skill_type"].unique().tolist()}')

assert len(mgr_records) == 102, f'Expected 102 manager records, got {len(mgr_records)}'
assert set(mgr_records['skill_name']) == {'N/A - Department-level analysis only'}, 'Unexpected skill name for Manager!'
assert set(mgr_records['skill_type']) == {'N/A'}, 'Unexpected skill type for Manager!'
print('\nCONFIRMED: All 102 Manager employees have exactly 1 record showing: "N/A - Department-level analysis only". No synthetic O*NET skills were fabricated.')

=== MANAGER ROLE EXCLUSION VERIFICATION ===
Expected Manager headcount: 102
Total Manager records in synthetic table: 102
Distinct skill names for Managers: ['N/A - Department-level analysis only']
Distinct skill types for Managers: ['N/A']

CONFIRMED: All 102 Manager employees have exactly 1 record showing: "N/A - Department-level analysis only". No synthetic O*NET skills were fabricated.


---
## Step 5 · Save CSV with Embedded Data Warning

In [5]:
out_csv = os.path.join(PROC, 'employee_skills_synthetic.csv')

warning_comment = (
    '# SYNTHETIC DATA — employee current-skill possession was not present in any source file '
    'and has been simulated using a tenure/training-based heuristic for MVP demonstration purposes only. '
    'This must NOT be presented to stakeholders as real observed skill data. Real deployment requires '
    'an actual skills inventory (HRIS export, LMS completion records, or self-assessment survey).\n'
)

# Write warning comment followed by CSV content
with open(out_csv, 'w', encoding='utf-8') as f:
    f.write(warning_comment)
    df_synthetic.to_csv(f, index=False)

file_size = os.path.getsize(out_csv)
print(f'Saved synthetic file to: {out_csv}')
print(f'File Size: {file_size:,} bytes')
print(f'Total Rows: {len(df_synthetic):,} (plus 1 comment line + 1 header line)')

# Verify round-trip reading with comment='# '
df_reloaded = pd.read_csv(out_csv, comment='#')
assert len(df_reloaded) == len(df_synthetic), 'Reloaded row count mismatch!'
assert list(df_reloaded.columns) == ['EmployeeNumber', 'skill_name', 'skill_type', 'has_skill']
print('\nCONFIRMED: Round-trip verification passed cleanly.')

Saved synthetic file to: ..\data\processed\employee_skills_synthetic.csv
File Size: 490,436 bytes
Total Rows: 13,782 (plus 1 comment line + 1 header line)

CONFIRMED: Round-trip verification passed cleanly.
